# 🏛️ Taller Final: Pipeline ETL y Análisis de Contratación Pública en PostgreSQL
### Extracción, Limpieza, Transformación, Modelado Relacional y Minería de Datos sobre Contratos Estatales (SECOP)

[![Python](https://img.shields.io/badge/Python-3.10%2B-blue.svg?logo=python&logoColor=white)](https://www.python.org/)
[![PostgreSQL](https://img.shields.io/badge/PostgreSQL-15%2B-336791.svg?logo=postgresql&logoColor=white)](https://www.postgresql.org/)
[![SQLAlchemy](https://img.shields.io/badge/SQLAlchemy-2.0%2B-D71F00.svg?logo=sqlalchemy&logoColor=white)](https://www.sqlalchemy.org/)
[![Plotly](https://img.shields.io/badge/Plotly-5.0%2B-3F4F75.svg?logo=plotly&logoColor=white)](https://plotly.com/)
[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Bernal-XFK/Pipeline-ETL-Contratacion-Publica-PostgreSQL/blob/main/Taller_Final_ETL_Contratacion_PostgreSQL.ipynb)
[![Proyecto Universitario](https://img.shields.io/badge/Proyecto-Universitario-informational.svg)](https://www.unicesmag.edu.co/)
[![License: MIT](https://img.shields.io/badge/License-MIT-green.svg)](LICENSE)

---

## 👨‍💻 Información Académica y Autoría
- **Estudiante:** **José Alejandro Bernal Figueroa** ([@Bernal-XFK](https://github.com/Bernal-XFK))
- **Institución:** **Universidad CESMAG**
- **Facultad:** Facultad de Ingeniería
- **Programa:** **Ingeniería de Sistemas**
- **Asignatura:** Aprendizaje Automático / Bases de Datos Avanzadas / Analítica de Datos
- **Docente:** Ing. Moya
- **Repositorio GitHub:** [Bernal-XFK/Pipeline-ETL-Contratacion-Publica-PostgreSQL](https://github.com/Bernal-XFK/Pipeline-ETL-Contratacion-Publica-PostgreSQL)

---



## CAPÍTULO 0 — Instalación y configuración del entorno
> *Preparando el taller: las herramientas que usaremos y por qué.*

---

Este capítulo se encarga de preparar nuestro entorno de trabajo en Google Colab. Instalará las librerías necesarias para la conexión a bases de datos, manipulación de datos, visualización interactiva y el manejo de mensajes. Esto asegura que todas las herramientas que utilizaremos a lo largo del taller estén disponibles y listas para su uso.

In [28]:
# Instalar dependencias necesarias
!pip install psycopg2-binary sqlalchemy plotly pandas numpy ipywidgets --quiet

In [29]:
import pandas as pd
import numpy as np
import plotly.express as px
import plotly.graph_objects as go
from plotly.subplots import make_subplots
from sqlalchemy import create_engine, text
from IPython.display import display, Markdown
import warnings
warnings.filterwarnings("ignore")

# Paleta institucional para consistencia visual
COLORES = {
    "entidad":  "#1A5276",   # Azul oscuro
    "sector":   "#E67E22",   # Naranja
    "acento":   "#2ECC71",   # Verde
    "alerta":   "#E74C3C",   # Rojo
    "neutro":   "#95A5A6"    # Gris
}

TEMPLATE_PLOTLY = "plotly_white"
print("Librerias cargadas correctamente.")

Librerias cargadas correctamente.


---

## CAPÍTULO 1 — Conexión a PostgreSQL
> *Abriendo la puerta a los datos: estableciendo la conexión con la base de datos de contratos.*

---

En este capítulo, estableceremos la conexión a nuestra base de datos PostgreSQL alojada en Aiven Cloud. Utilizaremos SQLAlchemy, una potente librería que nos permite interactuar con bases de datos relacionales de manera Pythonica, es decir, usando objetos de Python en lugar de escribir SQL directamente. Un 'engine' de conexión es como el motor que maneja toda la comunicación con la base de datos, traduciendo nuestras instrucciones de Python a SQL y viceversa. Finalmente, usamos `sslmode=require` para asegurar que nuestra conexión con Aiven Cloud sea segura, encriptando los datos que viajan entre nuestro notebook y el servidor. Conectarse a la base de datos es como abrir la puerta del archivo donde están guardados todos los contratos.

In [30]:
def crear_engine_postgres(user, passwd, host, port, dbname):
    """
    Crea y valida la conexión a PostgreSQL usando SQLAlchemy.
    """
    conn_str = f"postgresql+psycopg2://{user}:{passwd}@{host}:{port}/{dbname}?sslmode=require"
    engine = create_engine(conn_str, pool_pre_ping=True)
    # Verificar conexión
    with engine.begin() as conn:
        version = conn.execute(text("SELECT version()")).scalar()
    print(f" Conexion exitosa.\n Servidor: {version[:60]}...")
    return engine

In [31]:
import os

# Credenciales PostgreSQL (Aiven Cloud / Entorno Seguro)
DB_USER     = os.getenv('DB_USER', 'avnadmin')
DB_PASSWD   = os.getenv('DB_PASSWD', 'YOUR_DB_PASSWORD')
DB_HOST     = os.getenv('DB_HOST', 'pg-229fb88a-unicesmag-5619.a.aivencloud.com')
DB_PORT     = os.getenv('DB_PORT', '19060')
DB_NAME     = os.getenv('DB_NAME', 'defaultdb')

# Creación del engine SQLAlchemy
try:
    engine = crear_engine_postgres(DB_USER, DB_PASSWD, DB_HOST, DB_PORT, DB_NAME)
except Exception as e:
    print('Aviso: Defina la variable de entorno DB_PASSWD o proporcione sus credenciales seguras para ejecutar contra la base de datos en vivo:', e)


 Conexion exitosa.
 Servidor: PostgreSQL 17.9 on x86_64-pc-linux-gnu, compiled by gcc (GCC...


---

## CAPÍTULO 2 — Exploración inicial de la tabla raw
> *¿Qué hay en el archivo? Un primer vistazo a los contratos tal como vienen de la fuente.*

---

En esta etapa, nos adentramos en la 'zona raw' o de "staging". Esto significa que los datos se encuentran tal cual los recibimos de la fuente original, sin ninguna modificación, limpieza o validación de tipo de dato. Es crucial entender la estructura de estos datos crudos: cuántos registros existen, qué columnas tenemos, qué tipo de información contienen (aunque todo esté como texto inicialmente), y cómo se ven las primeras filas. Esta exploración inicial nos dará una base sólida para las transformaciones que realizaremos más adelante.

In [32]:
def explorar_tabla_raw(engine, tabla):
    """
    Realiza una exploración inicial de la tabla raw en PostgreSQL.
    """
    try:
        with engine.begin() as conn:
            # Total de filas
            total = conn.execute(text(f'SELECT COUNT(*) FROM "{tabla}"')).scalar()
            print(f"Total de registros en tabla raw: {total:,}")

            # Columnas y tipos
            cols_query = text("""
                SELECT column_name, data_type
                FROM information_schema.columns
                WHERE table_name = :tabla
                ORDER BY ordinal_position
            """)
            cols = pd.read_sql(cols_query, conn, params={"tabla": tabla})
            print(f"\n Columnas disponibles ({len(cols)} en total):")
            display(cols)

        # Muestra de 5 filas
        df_muestra = pd.read_sql(f'SELECT * FROM "{tabla}" LIMIT 5', engine)
        print("\n Primeras 5 filas:")
        display(df_muestra)

        return df_muestra.columns.tolist()
    except Exception as e:
        print(f" Error al explorar la tabla raw: {e}")
        return []

In [33]:
TABLA_RAW = "secop_contratos_santander" # Definido en el contexto técnico fijo
columnas_raw = explorar_tabla_raw(engine, TABLA_RAW)

Total de registros en tabla raw: 254,331

 Columnas disponibles (79 en total):


,column_name,data_type
0,nombre_entidad,character varying
1,nit_entidad,character varying
2,departamento,character varying
3,ciudad,character varying
4,localizaci_n,character varying
...,...,...
74,fecha_de_firma,character varying
75,ultima_actualizacion,character varying
76,fecha_inicio_liquidacion,character varying
77,fecha_fin_liquidacion,character varying



 Primeras 5 filas:


,nombre_entidad,nit_entidad,departamento,ciudad,localizaci_n,orden,sector,rama,entidad_centralizada,proceso_de_compra,...,nombre_ordenador_de_pago,tipo_de_documento_ordenador_de_pago,n_mero_de_documento_ordenador_de_pago,documentos_tipo,descripcion_documentos_tipo,fecha_de_firma,ultima_actualizacion,fecha_inicio_liquidacion,fecha_fin_liquidacion,fecha_de_notificaci_n_de_prorrogaci_n
0,ALCALDIA MUNICIPIO DE SAN GIL - SANTANDER,800099824,Santander,San Gil,"Colombia, Santander , San Gil",Territorial,No aplica/No pertenece,Ejecutivo,Centralizada,CO1.BDOS.4707823,...,HERMES ORTIZ RODRIGUEZ,Cédula de Ciudadanía,91071520,No,No definido,2023-07-10T00:00:00.000,2025-07-09T00:00:00.000,None,None,None
1,E.S.E. INSTITUTO DE SALUD DE BUCARAMANGA,800084206,Santander,No Definido,"Colombia, Santander , No Definido",Territorial,Salud y Protección Social,Corporación Autónoma,Centralizada,CO1.BDOS.4707570,...,No definido,No definido,No definido,No,No definido,2023-07-10T00:00:00.000,2023-11-14T00:00:00.000,None,None,None
2,INSTITUCION EDUCATIVA AURELIO MARTINEZ MUTIS,800249796,Santander,Bucaramanga,"Colombia, Santander , Bucaramanga",Territorial,Educación Nacional,Corporación Autónoma,Centralizada,CO1.BDOS.4535966,...,DORA BIBIANA SOLANO DALLOS,Cédula de Ciudadanía,63362142,No,No definido,2023-07-27T00:00:00.000,2023-09-17T00:00:00.000,None,None,None
3,UNIDAD PRESTADORA DE SALUD SANTANDER,901361090,Santander,Bucaramanga,"Colombia, Santander , Bucaramanga",Nacional,Salud y Protección Social,Ejecutivo,Descentralizada,CO1.BDOS.4612173,...,No definido,No definido,No definido,No,No definido,2023-07-13T00:00:00.000,2024-03-31T00:00:00.000,2024-05-11T00:00:00.000,2024-06-07T00:00:00.000,2024-02-11T00:00:00.000
4,MUNICIPIO DE BUCARAMANGA,890201222,Santander,Bucaramanga,"Colombia, Santander , Bucaramanga",Territorial,Servicio Público,Ejecutivo,Descentralizada,CO1.BDOS.500478,...,No definido,No definido,No definido,No,No definido,2018-08-10T00:00:00.000,2022-05-23T00:00:00.000,None,None,None


In [34]:
def identificar_entidad_x(engine, tabla, patron="ALCALD"):
    """
    Identifica la Entidad X.
    """
    try:
        query = text(f"""
            SELECT nombre_entidad, COUNT(*) as total_contratos
            FROM "{tabla}"
            WHERE UPPER(nombre_entidad) LIKE :patron
            GROUP BY nombre_entidad
            ORDER BY total_contratos DESC
            LIMIT 20
        """)
        with engine.begin() as conn:
            df = pd.read_sql(query, conn, params={"patron": f"%{patron}%"})

        print(f"Entidades que contienen '{patron}':")
        display(df)

        if not df.empty:
            entidad_x = df.iloc[0]["nombre_entidad"]
            print(f"\n Entidad X seleccionada (mas frecuente): '{entidad_x}'")
            return entidad_x
        else:
            print(f"Advertencia: No se encontro ninguna entidad que coincida con el patron '{patron}'.")
            return None
    except Exception as e:
        print(f"Error al identificar la Entidad X: {e}")
        return None

In [35]:
ENTIDAD_X_PATRON = "ALCALD" # Definido en el contexto técnico fijo
ENTIDAD_X = identificar_entidad_x(engine, TABLA_RAW, patron=ENTIDAD_X_PATRON)
# Si el nombre automático no es correcto, sobreescribe manualmente:
# ENTIDAD_X = "NOMBRE EXACTO DE LA ENTIDAD"
print(f"\n Entidad X confirmada: {ENTIDAD_X}")

Entidades que contienen 'ALCALD':


,nombre_entidad,total_contratos
0,ALCALDIA DISTRITAL BARRANCABERMEJA,28011
1,ALCALDIA DE FLORIDABLANCA,14757
2,ALCALDIA MUNICIPIO DE GIRON SANTANDER,6112
3,ALCALDIA MUNICIPIO DE SAN GIL - SANTANDER,3074
4,ALCALDÍA MUNICIPAL DE EL SOCORRO,1528
5,ALCALDIA MUNICIPIO DE RIONEGRO,980
6,ALCALDIA MUNICIPAL DE MALAGA,878
7,ALCALDIA MUNICIPIAL DE SIMACOTA,665
8,ALCALDiA MUNICIPIO DE CURITI,629
9,ALCALDIA MUNICIPAL DE BARBOSA SANTANDER,599



 Entidad X seleccionada (mas frecuente): 'ALCALDIA DISTRITAL BARRANCABERMEJA'

 Entidad X confirmada: ALCALDIA DISTRITAL BARRANCABERMEJA


In [36]:
def identificar_sector(engine, tabla, entidad_x):
    """
    Busca el sector al que pertenece la Entidad X.
    """
    if entidad_x is None:
        print("Entidad X no definida. No se puede identificar el sector.")
        return None, None

    candidatos = ["sector_entidad", "sector", "rama", "nivel_entidad", "tipo_entidad_compradora"]

    with engine.begin() as conn:
        cols_existentes = pd.read_sql(text(f'SELECT column_name FROM information_schema.columns WHERE table_name = :tabla'), conn, params={"tabla": tabla})["column_name"].tolist()

    campo_sector = next((c for c in candidatos if c in cols_existentes), "sector_entidad")

    try:
        query = text(f'SELECT "{campo_sector}", COUNT(*) as contratos FROM "{tabla}" WHERE nombre_entidad = :entidad GROUP BY "{campo_sector}" ORDER BY contratos DESC')
        with engine.begin() as conn:
            df = pd.read_sql(query, conn, params={"entidad": entidad_x})

        print(f"Sector(es) de '{entidad_x}':")
        display(df)

        if not df.empty:
            sector_x = df.iloc[0][campo_sector]
            print(f"\n Sector principal: '{sector_x}'")
            return sector_x, campo_sector
        else:
            print(f"No se encontro sector para la Entidad X '{entidad_x}'.")
            return None, campo_sector
    except Exception as e:
        print(f"Error al identificar el sector para la Entidad X: {e}")
        return None, campo_sector

In [37]:
SECTOR_X, CAMPO_SECTOR = identificar_sector(engine, TABLA_RAW, ENTIDAD_X)

Sector(es) de 'ALCALDIA DISTRITAL BARRANCABERMEJA':


,sector,contratos
0,Servicio Público,28011



 Sector principal: 'Servicio Público'


---

## CAPÍTULO 3 — PIPELINE ETL: Extracción y filtrado
> *Preparando los ingredientes: seleccionando los contratos que realmente importan para nuestro análisis.*

---

La primera fase de nuestro proceso ETL (Extracción, Transformación, Carga) es la **Extracción**. Aquí, no solo leeremos los datos de PostgreSQL, sino que aplicaremos un filtro inicial crucial: solo nos interesan los contratos que pertenecen al mismo sector que la Entidad X (nuestra Alcaldía de interés), y excluiremos aquellos registros que aún no son contratos formalizados. Esto se debe a que estados como "Borrador" o "En aprobación" representan procesos que aún no han concluido y podrían distorsionar nuestro análisis de contratos firmes. La siguiente sección mostrará cuántos registros se tenían inicialmente en el sector y cuántos quedan después de aplicar estos filtros, dándonos una idea clara del volumen de datos con el que trabajaremos.

In [68]:
def extraer_datos_sector(engine, tabla, entidad_x, sector_x, campo_sector, estados_excluir):
    """
    Extrae de la tabla raw todos los contratos del sector.
    """
    try:
        q_antes = text(f"""
            SELECT COUNT(*) FROM "{tabla}"
            WHERE "{campo_sector}" = :sector
        """)

        estados_sql = ", ".join([f"'{e}'" for e in estados_excluir])
        query_filtrada = f"""
            SELECT *
            FROM "{tabla}"
            WHERE "{campo_sector}" = :sector
              AND estado_contrato NOT IN ({estados_sql})
        """

        with engine.begin() as conn:
            total_antes = conn.execute(q_antes, {"sector": sector_x}).scalar()
            df_raw = pd.read_sql(text(query_filtrada), conn, params={"sector": sector_x})

        total_despues = len(df_raw)
        excluidos = total_antes - total_despues
        pct_excluido = round(excluidos / total_antes * 100, 2) if total_antes > 0 else 0

        print("Resultado del filtrado inicial:")
        print(f"   Registros en el sector (sin filtro de estado) : {total_antes:>10,}")
        print(f"   Registros excluidos (estados preliminares)    : {excluidos:>10,}  ({pct_excluido}%) ")
        print(f"   Registros conservados para el analisis        : {total_despues:>10,}")
        return df_raw, {"antes_filtro": total_antes, "despues_filtro": total_despues, "excluidos": excluidos, "pct_excluido": pct_excluido}
    except Exception as e:
        print(f"Error al extraer datos del sector: {e}")
        return pd.DataFrame(), {}

In [39]:
ESTADOS_EXCLUIR = ["Borrador", "En aprobacion", "Enviado Proveedor"]
df_raw, conteos_filtro = extraer_datos_sector(
    engine, TABLA_RAW,
    ENTIDAD_X, SECTOR_X, CAMPO_SECTOR, ESTADOS_EXCLUIR
)
print(f"\n  Dimensión del DataFrame extraído: {df_raw.shape[0]:,} filas × {df_raw.shape[1]} columnas")

Resultado del filtrado inicial:
   Registros en el sector (sin filtro de estado) :    115,272
   Registros excluidos (estados preliminares)    :      3,250  (2.82%) 
   Registros conservados para el analisis        :    112,022

  Dimensión del DataFrame extraído: 112,022 filas × 79 columnas


In [40]:
def graficar_conteo_filtrado(conteos, entidad_x):
    """Visualiza el impacto del filtro de estados con un gráfico de barras comparativo."""
    fig = go.Figure()
    fig.add_bar(
        x=["Antes del filtro", "Despues del filtro"],
        y=[conteos["antes_filtro"], conteos["despues_filtro"]],
        marker_color=[COLORES["neutro"], COLORES["entidad"]],
        text=[f"{conteos['antes_filtro']:,}", f"{conteos['despues_filtro']:,}"],
        textposition="outside"
    )
    fig.update_layout(
        title=f"Impacto del filtro de estados - Sector de {entidad_x}",
        yaxis_title="Numero de contratos",
        template=TEMPLATE_PLOTLY,
        showlegend=False
    )
    fig.show()

In [27]:
graficar_conteo_filtrado(conteos_filtro, ENTIDAD_X)

### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'El filtro de estados excluyó X contratos, lo que representa un Y% del total, dejando Z contratos válidos para el análisis. La Entidad X conserva X contratos y el resto del sector X.']

---

## CAPÍTULO 4 — PIPELINE ETL: Limpieza y transformación
> *Poniendo orden en la información: estandarizando y enriqueciendo los datos para el análisis.*

---

La etapa de **Limpieza y Transformación** es el corazón de nuestro pipeline ETL. Aquí, los datos extraídos de la zona raw son refinados para garantizar su calidad y utilidad analítica. Aplicaremos las siguientes transformaciones:

*   **Homologación de nombres de columnas**: A menudo, las fuentes de datos usan nombres inconsistentes. Estandarizaremos los nombres clave para facilitar el acceso y evitar errores.
*   **Limpieza de texto**: Eliminaremos espacios extra, unificaremos mayúsculas y minúsculas cuando sea necesario, y convertiremos valores como "sin información" o "N/A" en valores nulos (`NaN`) para un manejo adecuado.
*   **Conversión de fechas**: Las fechas, que suelen venir como texto, se transformarán a un formato de fecha real (`datetime`), y derivaremos columnas para el año y el mes, facilitando análisis temporales.
*   **Conversión de valores monetarios**: Los montos de los contratos se limpiarán de símbolos de moneda, separadores de miles y se convertirán a números (`float`), permitiendo realizar cálculos y agregaciones correctamente.
*   **Creación de grupo_comparacion**: Una columna fundamental para nuestro análisis será `grupo_comparacion`, que nos permitirá diferenciar claramente los contratos de la 'Entidad X' del resto del 'Sector sin Entidad X'.
*   **Validación de nulos y tipos**: Finalmente, realizaremos una verificación para asegurarnos de que los campos críticos no presenten una cantidad excesiva de valores faltantes y que los tipos de datos sean los esperados.

In [41]:
def homologar_columnas(df):
    """
    Mapea los nombres reales de columnas de la tabla raw a los nombres
    analíticos estandarizados que requiere el taller.
    """
    # Estandarización agresiva de nombres de columnas originales
    # Eliminamos espacios al inicio/final, pasamos a minúsculas y reemplazamos espacios internos por guiones bajos
    df.columns = [
        str(col).lower().strip().replace(' ', '_').replace('__', '_')
        for col in df.columns
    ]

    mapeo_candidatos = {
        "entidad":              ["nombre_entidad", "entidad", "nombre_entidad_compradora"],
        "sector":               ["sector_entidad", "sector", "rama", "nivel_entidad"],
        "estado_contrato":      ["estado_contrato", "estado_del_proceso", "estado"],
        "tipo_contrato":        ["tipo_contrato", "tipo_de_contrato"],
        "modalidad_contratacion": ["modalidad_de_contratacion", "modalidad_contratacion", "modalidad", "tipo_de_proceso"],
        "tipo_proveedor":       ["tipo_de_proveedor", "tipo_proveedor", "tipo_persona", "tipopersona", "tipodocproveedor"],
        "proveedor":            ["nombre_del_proveedor", "nombre_proveedor", "proveedor", "razon_social_proveedor"],
        "fecha_contrato":       ["fecha_de_firma_del_contrato", "fecha_firma", "fecha_contrato", "fecha_de_contrato", "fecha_inicio_contrato", "fecha_de_inicio_del_contrato"],
        "valor_contrato":       ["valor_del_contrato", "valor_contrato", "valor_total_contrato", "cuantia_contrato"],
    }

    cols_df = df.columns.tolist()
    renombrar = {}
    homologacion_log = []

    for nombre_analitico, candidatos in mapeo_candidatos.items():
        encontrado = next((c for c in candidatos if c in cols_df), None)
        if encontrado:
            renombrar[encontrado] = nombre_analitico
            homologacion_log.append({"campo_raw": encontrado, "campo_analitico": nombre_analitico, "estado": "✅ Ok"})
        else:
            df[nombre_analitico] = np.nan
            homologacion_log.append({"campo_raw": "NOT_FOUND", "campo_analitico": nombre_analitico, "estado": "☐ Creada como nula"})

    df = df.rename(columns=renombrar)
    print("⌒ Homologacin de columnas aplicada:")
    display(pd.DataFrame(homologacion_log))
    return df

In [42]:
def limpiar_textos(df):
    """
    Limpia columnas de texto:
    - Strip de espacios.
    - Reemplaza cadenas vacías y variaciones de nulos por NaN.
    - Estandariza a mayúsculas.
    """
    VACIOS = ["", "nan", "none", "null", "sin información", "sin informacion",
              "no aplica", "no registra", "nd", "n/a", "-"]

    # Columnas que esperamos procesar
    potential_cols = ["entidad", "sector", "estado_contrato", "tipo_contrato",
                      "modalidad_contratacion", "tipo_proveedor", "proveedor"]

    # Solo procesar las que existen actualmente en el df
    existing_cols = [col for col in potential_cols if col in df.columns]

    for col in existing_cols:
        df[col] = df[col].astype(str).str.strip()
        df[col] = df[col].replace(VACIOS, np.nan, regex=False)
        if col not in ["proveedor"]: # No estandarizamos a mayúsculas nombres de proveedores largos
            df[col] = df[col].str.upper()

    nulos_por_col = df[existing_cols].isnull().sum()
    print(" Nulos encontrados tras limpieza de texto:")
    display(nulos_por_col[nulos_por_col > 0].rename("nulos").to_frame())
    return df

In [43]:
def convertir_fechas(df):
    """
    Convierte la columna fecha_contrato de texto a datetime.
    Deriva las columnas 'anio' (entero) y 'mes' (entero 1-12).
    Muestra cuántos registros no pudieron parsearse.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con los datos de contrato.

    Retorna
    -------
    pd.DataFrame
        DataFrame con la columna de fecha convertida y nuevas columnas de año y mes.
    """
    if "fecha_contrato" not in df.columns:
        print("  Columna 'fecha_contrato' no encontrada. No se realizará la conversión de fechas.")
        # Crear columnas 'anio' y 'mes' como NaN para evitar errores posteriores
        df["anio"] = pd.NA
        df["mes"] = pd.NA
        return df

    df["fecha_contrato"] = pd.to_datetime(df["fecha_contrato"], errors="coerce", dayfirst=True)

    no_parseados = df["fecha_contrato"].isnull().sum()

    df["anio"] = df["fecha_contrato"].dt.year.astype("Int64") # Usar Int64 para admitir nulos
    df["mes"]  = df["fecha_contrato"].dt.month.astype("Int64")

    print(f" Fechas convertidas correctamente.")
    print(f"   Registros sin fecha válida : {no_parseados:,}")

    # Mostrar rango de fechas solo si hay fechas válidas
    if df['fecha_contrato'].min() is not pd.NaT and df['fecha_contrato'].max() is not pd.NaT:
        print(f"   Rango de fechas detectado  : {df['fecha_contrato'].min().date()} → {df['fecha_contrato'].max().date()}")
    else:
        print("   Rango de fechas no disponible (no hay fechas válidas).")

    # Mostrar años únicos solo si hay años válidos
    unique_years = sorted(df['anio'].dropna().unique().tolist()) if not df['anio'].dropna().empty else []
    print(f"   Años únicos en los datos   : {unique_years}")
    return df

In [44]:
def convertir_valores(df):
    """
    Convierte la columna valor_contrato de texto a float.
    Elimina separadores de miles, símbolos de moneda y espacios.
    Valida que no hayan valores negativos o cero.
    Genera estadísticas básicas de validación.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con los datos de contrato.

    Retorna
    -------
    pd.DataFrame
        DataFrame con la columna de valor de contrato convertida a numérico.
    """
    if "valor_contrato" not in df.columns:
        print("  Columna 'valor_contrato' no encontrada. No se realizará la conversión de valores monetarios.")
        df["valor_contrato_limpio"] = np.nan # Crear columna como NaN para evitar errores posteriores
        return df

    # Convertir a string para aplicar métodos de string y reemplazar valores no numéricos
    df["valor_contrato_limpio"] = (
        df["valor_contrato"]
        .astype(str)
        .str.replace(r"[.,]", "", regex=True) # Eliminar separadores de miles y decimales
        .str.replace(r"[^\d-]", "", regex=True) # Eliminar cualquier caracter no dígito o guion (para negativos)
        .str.strip()
        .replace("", np.nan) # Reemplazar cadenas vacías (resultantes de la limpieza) con NaN
    )
    # Convertir a numérico, forzando errores a NaN
    df["valor_contrato_limpio"] = pd.to_numeric(df["valor_contrato_limpio"], errors="coerce")

    # Reemplazar ceros y negativos por NaN (no representan valor real para análisis de monto)
    df.loc[df["valor_contrato_limpio"] <= 0, "valor_contrato_limpio"] = np.nan

    # Estadísticas de validación
    validos   = df["valor_contrato_limpio"].notna().sum()
    nulos_val = df["valor_contrato_limpio"].isna().sum()

    print(f" Conversión de valores monetarios:")
    print(f"   Registros con valor válido  : {validos:,}")
    print(f"   Registros sin valor válido  : {nulos_val:,}")

    if validos > 0:
        print(f"   Valor mínimo                : ${df['valor_contrato_limpio'].min():,.0f}")
        print(f"   Valor máximo                : ${df['valor_contrato_limpio'].max():,.0f}")
        print(f"   Valor promedio              : ${df['valor_contrato_limpio'].mean():,.0f}")
    else:
        print("   No hay valores monetarios válidos para mostrar estadísticas.")
    return df

In [45]:
def crear_grupo_comparacion(df, entidad_x):
    """
    Crea la columna 'grupo_comparacion' que diferencia:
    - 'Entidad X'          → contratos de la Alcaldía estudiada.
    - 'Sector sin Entidad X' → contratos del resto del sector.

    Esta columna es clave para todas las comparaciones del taller.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con los datos de contrato.
    entidad_x : str
        Nombre de la Entidad X.

    Retorna
    -------
    pd.DataFrame
        DataFrame con la nueva columna 'grupo_comparacion'.
    """
    if "entidad" not in df.columns:
        print("  Columna 'entidad' no encontrada. No se puede crear el grupo de comparación.")
        df["grupo_comparacion"] = "Indefinido"
        return df

    df["grupo_comparacion"] = np.where(
        df["entidad"] == entidad_x,
        "Entidad X",
        "Sector sin Entidad X"
    )

    dist = df["grupo_comparacion"].value_counts()
    print("👥 Distribución del grupo de comparación:")
    display(dist.rename("contratos").to_frame())
    return df

In [46]:
def transformar_datos(df_raw, entidad_x):
    """
    Orquesta todas las transformaciones en secuencia.
    Retorna el DataFrame listo para cargar en la tabla analítica.

    Parámetros
    ----------
    df_raw : pd.DataFrame
        DataFrame raw extraído de la base de datos.
    entidad_x : str
        Nombre de la Entidad X.

    Retorna
    -------
    pd.DataFrame
        DataFrame transformado y listo para el análisis.
    """
    print("=" * 60)
    print(" INICIANDO PIPELINE DE TRANSFORMACIÓN")
    print("=" * 60)

    df = df_raw.copy()

    print("\n[1/5] Homologando columnas...")
    df = homologar_columnas(df)

    print("\n[2/5] Limpiando textos...")
    df = limpiar_textos(df)

    print("\n[3/5] Convirtiendo fechas...")
    df = convertir_fechas(df)

    print("\n[4/5] Convirtiendo valores monetarios...")
    df = convertir_valores(df)

    print("\n[5/5] Creando grupo de comparación...")
    df = crear_grupo_comparacion(df, entidad_x)

    # Seleccionar únicamente los campos analíticos mínimos requeridos
    CAMPOS_ANALITICOS = [
        "entidad", "sector", "grupo_comparacion", "estado_contrato",
        "tipo_contrato", "modalidad_contratacion", "tipo_proveedor",
        "proveedor", "fecha_contrato", "anio", "mes",
        "valor_contrato_limpio"
    ]

    campos_disponibles = [c for c in CAMPOS_ANALITICOS if c in df.columns]
    campos_faltantes   = [c for c in CAMPOS_ANALITICOS if c not in df.columns]

    if campos_faltantes:
        print(f"\n  Campos no encontrados en el DataFrame final (requieren revisión): {campos_faltantes}")

    df_analitico = df[campos_disponibles].copy()

    print(f"\n Transformación completada.")
    print(f"   Dimensión tabla analítica: {df_analitico.shape[0]:,} filas × {df_analitico.shape[1]} columnas")
    return df_analitico

In [47]:
# Ejecutar la transformación con las funciones actualizadas (homologar_columnas, etc.)
df_analitico = transformar_datos(df_raw, ENTIDAD_X)
display(df_analitico.head(3))

 INICIANDO PIPELINE DE TRANSFORMACIÓN

[1/5] Homologando columnas...
⌒ Homologacin de columnas aplicada:


,campo_raw,campo_analitico,estado
0,nombre_entidad,entidad,✅ Ok
1,sector,sector,✅ Ok
2,estado_contrato,estado_contrato,✅ Ok
3,tipo_de_contrato,tipo_contrato,✅ Ok
4,modalidad_de_contratacion,modalidad_contratacion,✅ Ok
5,tipodocproveedor,tipo_proveedor,✅ Ok
6,NOT_FOUND,proveedor,☐ Creada como nula
7,fecha_de_inicio_del_contrato,fecha_contrato,✅ Ok
8,valor_del_contrato,valor_contrato,✅ Ok



[2/5] Limpiando textos...
 Nulos encontrados tras limpieza de texto:


,nulos
proveedor,112022



[3/5] Convirtiendo fechas...
 Fechas convertidas correctamente.
   Registros sin fecha válida : 69,837
   Rango de fechas detectado  : 2017-01-02 → 2026-12-03
   Años únicos en los datos   : [2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024, 2025, 2026]

[4/5] Convirtiendo valores monetarios...
 Conversión de valores monetarios:
   Registros con valor válido  : 110,368
   Registros sin valor válido  : 1,654
   Valor mínimo                : $1
   Valor máximo                : $9,000,000,000,000
   Valor promedio              : $398,439,788

[5/5] Creando grupo de comparación...
👥 Distribución del grupo de comparación:


,contratos
grupo_comparacion,
Sector sin Entidad X,84917
Entidad X,27105



 Transformación completada.
   Dimensión tabla analítica: 112,022 filas × 12 columnas


,entidad,sector,grupo_comparacion,estado_contrato,tipo_contrato,modalidad_contratacion,tipo_proveedor,proveedor,fecha_contrato,anio,mes,valor_contrato_limpio
0,ALCALDIA DISTRITAL BARRANCABERMEJA,SERVICIO PÚBLICO,Entidad X,CERRADO,PRESTACIÓN DE SERVICIOS,CONTRATACIÓN DIRECTA,CÉDULA DE CIUDADANÍA,NaN,2023-05-07,2023,5,16940000.0
1,MUNICIPIO DE BUCARAMANGA,SERVICIO PÚBLICO,Sector sin Entidad X,MODIFICADO,PRESTACIÓN DE SERVICIOS,CONTRATACIÓN DIRECTA,CÉDULA DE CIUDADANÍA,NaN,NaT,<NA>,<NA>,10200000.0
2,ALCALDIA DISTRITAL BARRANCABERMEJA,SERVICIO PÚBLICO,Entidad X,MODIFICADO,PRESTACIÓN DE SERVICIOS,CONTRATACIÓN DIRECTA,CÉDULA DE CIUDADANÍA,NaN,2023-04-07,2023,4,12500000.0


---

## CAPÍTULO 5 — PIPELINE ETL: Carga de la tabla analítica
> *Almacenando el producto final: guardando los datos limpios en su lugar definitivo.*

---

Hemos llegado a la etapa final de nuestro Pipeline ETL: la **Carga** (Load). En este punto, los datos ya han sido extraídos de la fuente raw, filtrados para incluir solo la información relevante, y transformados para asegurar su calidad y formato adecuado para el análisis. Ahora, el `df_analitico` que hemos construido se cargará en una nueva tabla dentro de nuestra base de datos PostgreSQL, la cual hemos nombrado `secop_contratos_entidad_x`. Esta tabla será la fuente principal para todos los análisis descriptivos que realizaremos a continuación.

Utilizaremos la opción `if_exists='replace'` al cargar los datos. Esto significa que si la tabla `secop_contratos_entidad_x` ya existe en la base de datos, será eliminada y recreada con los nuevos datos. Esta estrategia asegura que cada vez que ejecutemos el pipeline, siempre tengamos una versión actualizada y consistente de nuestra tabla analítica, aunque implica que cualquier dato preexistente en esa tabla se perderá. Finalmente, realizaremos algunas validaciones post-carga para confirmar que la información se almacenó correctamente y que cumple con las expectativas de estructura y contenido.

In [48]:
def cargar_tabla_analitica(df_analitico, engine, nombre_tabla):
    """Carga el DataFrame analitico en PostgreSQL."""
    try:
        print(f"Cargando {df_analitico.shape[0]:,} registros en '{nombre_tabla}'...")
        df_analitico.to_sql(nombre_tabla, engine, if_exists="replace", index=False, method="multi", chunksize=5000)

        with engine.begin() as conn:
            total_cargado = conn.execute(text(f'SELECT COUNT(*) FROM "{nombre_tabla}"')).scalar()

        if total_cargado == len(df_analitico):
            print(f"Carga exitosa: {total_cargado:,} registros en '{nombre_tabla}'.")
        else:
            print(f"Discrepancia: se esperaban {len(df_analitico):,}, se cargaron {total_cargado:,}.")
        return total_cargado
    except Exception as e:
        print(f"Error al cargar la tabla analitica '{nombre_tabla}': {e}")
        return 0

In [49]:
TABLA_ANALITICA = "secop_contratos_entidad_x" # Definido en el contexto técnico fijo
total_cargado = cargar_tabla_analitica(df_analitico, engine, TABLA_ANALITICA)

Cargando 112,022 registros en 'secop_contratos_entidad_x'...
Carga exitosa: 112,022 registros en 'secop_contratos_entidad_x'.


In [50]:
def validar_tabla_analitica(engine, tabla_analitica, entidad_x):
    """
    Ejecuta validaciones básicas sobre la tabla analítica recién creada:
    - Conteo total
    - Conteo por grupo_comparacion
    - Tipos de dato inferidos
    - Porcentaje de nulos por campo crítico
    - Rango de años y valores

    Parámetros
    ----------
    engine : sqlalchemy.Engine
        Motor de conexión a la base de datos.
    tabla_analitica : str
        Nombre de la tabla analítica en PostgreSQL.
    entidad_x : str
        Nombre de la Entidad X (para validación de grupos).

    Retorna
    -------
    pd.DataFrame
        DataFrame con todos los datos de la tabla analítica para su uso posterior.
    """
    try:
        # Leer la tabla completa para validaciones
        df_val = pd.read_sql(f'SELECT * FROM "{tabla_analitica}"', engine)

        print(" VALIDACIONES DE LA TABLA ANALÍTICA\n" + "="*45)

        print(f"\n1️  Total de registros  : {len(df_val):,}")

        if "grupo_comparacion" in df_val.columns:
            print("\n2️  Registros por grupo de comparación:")
            display(df_val["grupo_comparacion"].value_counts().rename("contratos").to_frame())
        else:
            print("  Columna 'grupo_comparacion' no encontrada para validación.")

        print("\n3️  Tipos de dato en la tabla analítica (inferidos por Pandas):")
        display(df_val.dtypes.rename("tipo").to_frame())

        print("\n4️  Porcentaje de nulos por campo crítico:")
        campos_criticos = ["entidad","grupo_comparacion","estado_contrato","tipo_contrato",
                           "modalidad_contratacion","tipo_proveedor","fecha_contrato",
                           "anio","mes","valor_contrato_limpio"]

        # Filtrar campos_criticos que realmente existan en df_val
        campos_criticos_existentes = [c for c in campos_criticos if c in df_val.columns]

        if campos_criticos_existentes:
            nulos_pct = (df_val[campos_criticos_existentes].isnull().sum() / len(df_val) * 100).round(2)
            display(nulos_pct.rename("% nulos").to_frame())
        else:
            print("  No se encontraron campos críticos para validar nulos.")

        if "anio" in df_val.columns and not df_val["anio"].dropna().empty:
            print(f"\n5️  Rango de años: {df_val['anio'].min()} → {df_val['anio'].max()}")
        else:
            print("  No se encontraron años válidos para determinar el rango.")

        if "valor_contrato_limpio" in df_val.columns and not df_val["valor_contrato_limpio"].dropna().empty:
            print(f"   Valor mínimo contrato : ${df_val['valor_contrato_limpio'].min():,.0f}")
            print(f"   Valor máximo contrato : ${df_val['valor_contrato_limpio'].max():,.0f}")
        else:
            print("  No se encontraron valores de contrato válidos para determinar el rango.")

        return df_val
    except Exception as e:
        print(f" Error al validar la tabla analítica '{tabla_analitica}': {e}")
        return pd.DataFrame()

In [51]:
df = validar_tabla_analitica(engine, TABLA_ANALITICA, ENTIDAD_X)
# df será el DataFrame principal para todos los análisis siguientes

 VALIDACIONES DE LA TABLA ANALÍTICA

1️  Total de registros  : 112,022

2️  Registros por grupo de comparación:


,contratos
grupo_comparacion,
Sector sin Entidad X,84917
Entidad X,27105



3️  Tipos de dato en la tabla analítica (inferidos por Pandas):


,tipo
entidad,object
sector,object
grupo_comparacion,object
estado_contrato,object
tipo_contrato,object
modalidad_contratacion,object
tipo_proveedor,object
proveedor,object
fecha_contrato,datetime64[ns]
anio,float64



4️  Porcentaje de nulos por campo crítico:


,% nulos
entidad,0.00
grupo_comparacion,0.00
estado_contrato,0.00
tipo_contrato,0.00
modalidad_contratacion,0.00
tipo_proveedor,0.00
fecha_contrato,62.34
anio,62.34
mes,62.34
valor_contrato_limpio,1.48



5️  Rango de años: 2017.0 → 2026.0
   Valor mínimo contrato : $1
   Valor máximo contrato : $9,000,000,000,000


---

## CAPÍTULO 6 — PREGUNTA 1: Resumen del Pipeline ETL
> *¿Qué hemos hecho y por qué? La historia completa de nuestros datos, desde la fuente hasta el análisis.*

---

En este capítulo, daremos respuesta formal a la primera pregunta del taller. Describiremos en detalle el pipeline ETL que hemos construido, explicando cada una de sus fases: desde la extracción inicial de los datos de la fuente raw en PostgreSQL, pasando por los criterios de filtrado aplicados, las decisiones técnicas de limpieza y transformación de los datos, hasta la carga final en la tabla analítica. El objetivo es proporcionar una visión clara y concisa de cómo hemos preparado los datos para el análisis, asegurando que están limpios, estructurados y listos para extraer conclusiones. Incluiremos una tabla resumen que visualice el flujo de trabajo y la magnitud de los cambios en cada etapa.

In [52]:
def mostrar_resumen_pipeline(conteos_filtro, df, entidad_x):
    """Genera una tabla visual que resume las etapas del pipeline ETL."""
    resumen = pd.DataFrame([
        {"Etapa": "Extraccion",     "Descripcion": "Lectura desde PostgreSQL tabla raw",
         "Resultado": f"{conteos_filtro['antes_filtro']:,} registros del sector"},
        {"Etapa": "Filtrado",       "Descripcion": "Exclusion estados: Borrador, En aprobacion, Enviado Proveedor",
         "Resultado": f"{conteos_filtro['excluidos']:,} excluidos ({conteos_filtro['pct_excluido']}%)"},
        {"Etapa": "Limpieza",       "Descripcion": "Textos, vacios, inconsistencias",
         "Resultado": f"Campos estandarizados"},
        {"Etapa": "Transformacion", "Descripcion": "Fechas -> datetime, valores -> float, grupo_comparacion",
         "Resultado": f"12 campos analiticos creados"},
        {"Etapa": "Carga",          "Descripcion": "Tabla analitica en PostgreSQL",
         "Resultado": f"{len(df):,} registros en secop_contratos_entidad_x"},
    ])

    fig = go.Figure(data=[go.Table(
        header=dict(values=list(resumen.columns),
                    fill_color=COLORES["entidad"], font=dict(color="white", size=13),
                    align="left"),
        cells=dict(values=[resumen[c].tolist() for c in resumen.columns],
                   fill_color=[["#EBF5FB", "#D6EAF8"] * 3],
                   align="left", font=dict(size=12))
    )])
    fig.update_layout(title="Resumen del Pipeline ETL - SECOP II Santander", height=300)
    fig.show()

### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'La Entidad X contrató un 35% más en 2023 que en 2022, mientras que el sector creció solo un 12%...']

---

## CAPÍTULO 7 — PREGUNTA 2: Comportamiento en monto y volumen frente al sector
> *¿Cuánto y cuántos? Un comparativo de la actividad contractual de la Entidad X con su entorno.*

---

En este capítulo, abordaremos la Pregunta 2 del taller, que se enfoca en el comportamiento económico y de volumen de la contratación. Analizaremos la evolución anual del valor total contratado (monto) y el número de contratos (volumen), estableciendo una comparación clave entre la Entidad X y el resto del sector al que pertenece. Estas comparaciones nos permitirán entender si la Entidad X sigue las tendencias del sector, si su actividad contractual crece o disminuye, y si hay diferencias significativas en su nivel de operación. Cada gráfica nos mostrará un aspecto diferente de esta dinámica y al final, buscaremos patrones y hallazgos relevantes para la planeación contractual.

In [53]:
# El error 'KeyError' ocurre porque el pipeline de transformaci3n fall3 previamente,
# dejando el DataFrame 'df' vac3o o sin las columnas procesadas.

if not df.empty and "grupo_comparacion" in df.columns:
    df_entidad = df[df["grupo_comparacion"] == "Entidad X"].copy()
    df_sector  = df[df["grupo_comparacion"] == "Sector sin Entidad X"].copy()

    print(f"1 Entidad X       : {len(df_entidad):,} contratos")
    print(f"1 Sector sin E.X. : {len(df_sector):,} contratos")
else:
    print(" Error: El DataFrame 'df' no contiene los datos transformados.")
    print(" Por favor, aseg3rate de ejecutar exitosamente la celda de 'transformar_datos' y 'cargar_tabla_analitica' antes de continuar.")

1 Entidad X       : 27,105 contratos
1 Sector sin E.X. : 84,917 contratos


In [69]:
def graficar_monto_anual(df, colores, template):
    """
    Gráfica de barras agrupadas: monto total contratado por año,
    comparando Entidad X vs Sector sin Entidad X.
    """
    # 1. Validación de datos previos
    if df.empty or 'anio' not in df.columns or df['anio'].isna().all():
        print(" Error: No hay datos de 'año' válidos para graficar.")
        print(" Por favor, asegúrate de haber ejecutado las celdas de transformación y validación (Capítulo 4 y 5) correctamente.")
        return pd.DataFrame()

    # Asegurar que 'anio' es numérico y manejar nulos
    df_plot = df.dropna(subset=['anio', 'valor_contrato_limpio']).copy()
    df_plot['anio'] = df_plot['anio'].astype(int)

    agg = (df_plot.groupby(['anio', "grupo_comparacion"])["valor_contrato_limpio"]
             .sum().reset_index()
             .rename(columns={"valor_contrato_limpio": "monto_total"}))

    if agg.empty:
        print(" Advertencia: El DataFrame resultante para la gráfica está vacío después del agrupamiento.")
        return agg

    agg["monto_miles_millones"] = agg["monto_total"] / 1e9

    fig = px.bar(
        agg,
        x="anio",
        y="monto_miles_millones",
        color="grupo_comparacion",
        barmode="group",
        color_discrete_map={"Entidad X": colores["entidad"],
                             "Sector sin Entidad X": colores["sector"]},
        title="Monto total contratado por año (Miles de millones COP)",
        labels={
            "monto_miles_millones": "Monto (miles de millones COP)",
            "anio": "Año",
            "grupo_comparacion": "Grupo"
        },
        text_auto=".1f",
        template=template
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(xaxis=dict(tickmode='linear', dtick=1))
    fig.show()
    return agg

agg_monto = graficar_monto_anual(df, COLORES, TEMPLATE_PLOTLY)

### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'En el año 2023, la Alcaldia tuvo su mayor volumen de contratación con Y miles de millones de COP, mientras que el sector mantuvo una estabilidad relativa.']

In [55]:
def graficar_contratos_anuales(df, colores, template):
    """
    Gráfica de líneas: número de contratos por año por grupo.
    Permite ver si la cantidad de contratos aumenta o disminuye en el tiempo.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con los datos de contrato (debe incluir 'anio', 'grupo_comparacion').
    colores : dict
        Diccionario de colores para la visualización.
    template : str
        Plantilla de Plotly para el estilo del gráfico.

    Retorna
    -------
    pd.DataFrame
        DataFrame agregado con el número de contratos por año y grupo.
    """
    # Asegurar que 'anio' es numérico y manejar nulos
    df_plot = df.dropna(subset=['anio']).copy()
    df_plot['anio'] = df_plot['anio'].astype(int)

    agg = (df_plot.groupby([df_plot.anio, "grupo_comparacion"])
             .size().reset_index(name="num_contratos"))

    fig = px.line(
        agg,
        x="anio",
        y="num_contratos",
        color="grupo_comparacion",
        markers=True,
        color_discrete_map={"Entidad X": colores["entidad"],
                             "Sector sin Entidad X": colores["sector"]},
        title="Número de contratos por año",
        labels={
            "num_contratos": "Número de contratos",
            "anio": "Año",
            "grupo_comparacion": "Grupo"
        },
        template=template
    )
    fig.update_traces(line=dict(width=3), marker=dict(size=8))
    fig.update_layout(xaxis=dict(tickmode='linear', dtick=1)) # Asegura que todos los años se muestren
    fig.show()
    return agg

agg_num = graficar_contratos_anuales(df, COLORES, TEMPLATE_PLOTLY)

### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'La Entidad X ha mantenido un número de contratos relativamente estable, mientras que el sector mostró un pico en el año Y, lo que sugiere una mayor agilidad o necesidad de contratación por parte del sector en ese periodo.']

In [56]:
def graficar_por_tipo_contrato(df, colores, template):
    """
    Dos gráficas de barras horizontales:
    1. Monto total por tipo de contrato y grupo.
    2. Número de contratos por tipo de contrato y grupo.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con los datos de contrato (debe incluir 'tipo_contrato', 'grupo_comparacion', 'valor_contrato_limpio').
    colores : dict
        Diccionario de colores para la visualización.
    template : str
        Plantilla de Plotly para el estilo del gráfico.

    Retorna
    -------
    pd.DataFrame
        DataFrame agregado con monto y número de contratos por tipo y grupo.
    """
    agg = (df.groupby(["tipo_contrato", "grupo_comparacion"])
             .agg(monto=("valor_contrato_limpio", "sum"),
                  contratos=("valor_contrato_limpio", "count"))
             .reset_index())
    agg["monto_miles_millones"] = agg["monto"] / 1e9

    # Gráfica 1: Monto
    fig1 = px.bar(
        agg.sort_values("monto_miles_millones", ascending=True),
        y="tipo_contrato",
        x="monto_miles_millones",
        color="grupo_comparacion",
        barmode="group",
        orientation="h",
        color_discrete_map={"Entidad X": colores["entidad"],
                             "Sector sin Entidad X": colores["sector"]},
        title="Monto total por tipo de contrato (miles de millones COP)",
        labels={
            "monto_miles_millones": "Monto (miles de millones)",
            "tipo_contrato": "Tipo de contrato"
        },
        template=template
    )
    fig1.update_layout(yaxis=dict(autorange="reversed")) # Asegura que el tipo de contrato más grande esté arriba
    fig1.show()

    # Gráfica 2: Número de contratos
    fig2 = px.bar(
        agg.sort_values("contratos", ascending=True),
        y="tipo_contrato",
        x="contratos",
        color="grupo_comparacion",
        barmode="group",
        orientation="h",
        color_discrete_map={"Entidad X": colores["entidad"],
                             "Sector sin Entidad X": colores["sector"]},
        title="Número de contratos por tipo de contrato",
        labels={
            "contratos": "Número de contratos",
            "tipo_contrato": "Tipo de contrato"
        },
        template=template
    )
    fig2.update_layout(yaxis=dict(autorange="reversed")) # Asegura que el tipo de contrato más grande esté arriba
    fig2.show()
    return agg

agg_tipo = graficar_por_tipo_contrato(df, COLORES, TEMPLATE_PLOTLY)

### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'La Entidad X y el sector comparten los mismos tipos de contrato predominantes, con el contrato de X concentrando el Y% del monto total. Sin embargo, la Entidad X muestra una mayor proporción de contratos de Z, lo que podría indicar una necesidad específica.']

In [57]:
def tabla_comparativa_resumen(df):
    """
    Genera una tabla interactiva comparando métricas clave
    entre la Entidad X y el Sector sin Entidad X.
    """
    # 1. Asegurar que la columna de comparación existe en el DataFrame
    if "grupo_comparacion" not in df.columns:
        df["grupo_comparacion"] = np.where(df["entidad"] == ENTIDAD_X, "Entidad X", "Sector sin Entidad X")

    # 2. Definir explícitamente los subsets para esta visualización
    df_e = df[df["grupo_comparacion"] == "Entidad X"]
    df_s = df[df["grupo_comparacion"] == "Sector sin Entidad X"]

    def metricas(d, grupo):
        # Verificación de datos para evitar errores de cálculo
        val_col = 'valor_contrato_limpio'
        has_data = not d.empty and val_col in d.columns and not d[val_col].dropna().empty

        if not has_data:
            return {
                "Grupo": grupo,
                "Total contratos": f"{len(d):,}",
                "Monto total (COP)": "$0",
                "Monto promedio (COP)": "$0",
                "Monto mediano (COP)": "$0",
                "Años cubiertos": "N/A"
            }
        else:
            return {
                "Grupo": grupo,
                "Total contratos": f"{len(d):,}",
                "Monto total (COP)": f"${d[val_col].sum():,.0f}",
                "Monto promedio (COP)": f"${d[val_col].mean():,.0f}",
                "Monto mediano (COP)": f"${d[val_col].median():,.0f}",
                "Años cubiertos": f"{int(d['anio'].min())} – {int(d['anio'].max())}" if 'anio' in d.columns and not d['anio'].dropna().empty else "N/A"
            }

    # Construir la tabla
    tabla = pd.DataFrame([
        metricas(df_e, "Entidad X"),
        metricas(df_s, "Sector sin Entidad X")
    ])

    # Visualización
    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns),
                    fill_color=COLORES["entidad"], font=dict(color="white", size=12),
                    align="center"),
        cells=dict(values=[tabla[c].tolist() for c in tabla.columns],
                   fill_color=[["#EBF5FB", "#D6EAF8"]],
                   align="center", font=dict(size=12))
    )])

    fig.update_layout(title="Tabla comparativa: Entidad X vs Sector sin Entidad X", height=250)
    fig.show()

tabla_comparativa_resumen(df)

### 🔍 Interpretación Pregunta 2:
[Celda que el equipo debe completar con sus hallazgos: ¿La Entidad X contrata más o menos que el promedio del sector? ¿Ha aumentado o disminuido su contratación? ¿Qué tipos de contrato son predominantes?]

---

## CAPÍTULO 8 — PREGUNTA 3: Estilo de contratación
> *¿Cómo compra la Entidad X? Un análisis de modalidades y proveedores.*

---

El 'estilo de contratación' de una entidad revela mucho sobre su operación, sus riesgos y su eficiencia. En este capítulo, responderemos a la Pregunta 3 del taller, analizando dos aspectos clave:

*   **Modalidades de contratación**: ¿Utiliza predominantemente la licitación pública (más compleja pero transparente), o recurre con frecuencia a la contratación directa (más rápida pero con riesgos inherentes)? Las proporciones de cada modalidad nos darán una idea de la carga administrativa y el nivel de competencia promovido.
*   **Tipo de proveedor**: ¿Contrata mayormente con personas naturales (servicios profesionales, consultorías) o con personas jurídicas (empresas, sociedades)? Un alto porcentaje de contratos con personas naturales podría indicar una dependencia en servicios personales o una estructura operativa específica.

Estas métricas nos permitirán identificar patrones y posibles oportunidades de mejora en la gestión contractual de la Entidad X.

In [58]:
def graficar_modalidades(df, colores, template):
    """
    Barras horizontales que muestran la distribución porcentual
    de contratos por modalidad de contratación en cada grupo.
    Permite comparar si la Entidad X usa más contratación directa que el sector.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con los datos de contrato (debe incluir 'modalidad_contratacion', 'grupo_comparacion').
    colores : dict
        Diccionario de colores para la visualización.
    template : str
        Plantilla de Plotly para el estilo del gráfico.

    Retorna
    -------
    pd.DataFrame
        DataFrame agregado con la distribución porcentual por modalidad y grupo.
    """
    agg = (df.groupby(["modalidad_contratacion", "grupo_comparacion"])
             .size().reset_index(name="contratos"))

    # Calcular porcentaje dentro de cada grupo
    totales = df.groupby("grupo_comparacion").size().rename("total")
    agg = agg.merge(totales, on="grupo_comparacion")
    agg["pct"] = (agg["contratos"] / agg["total"] * 100).round(2)

    fig = px.bar(
        agg.sort_values("pct", ascending=True),
        y="modalidad_contratacion", x="pct",
        color="grupo_comparacion",
        barmode="group", orientation="h",
        color_discrete_map={"Entidad X": colores["entidad"],
                             "Sector sin Entidad X": colores["sector"]},
        title="Distribución porcentual por modalidad de contratación",
        labels={"pct": "% de contratos", "modalidad_contratacion": "Modalidad"},
        text=agg.sort_values("pct", ascending=True)["pct"].apply(lambda x: f"{x:.1f}%"),
        template=template
    )
    fig.update_traces(textposition="outside")
    fig.update_layout(yaxis=dict(autorange="reversed"))
    fig.show()
    return agg

agg_modalidad = graficar_modalidades(df, COLORES, TEMPLATE_PLOTLY)

### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'La Entidad X recurre en mayor medida a la modalidad de X (Y% de sus contratos) que el resto del sector (Z%), lo que podría indicar una necesidad de revisar los procesos para fomentar la competencia o simplificar la gestión.']

In [59]:
def graficar_tipo_proveedor(df, colores, template):
    """
    Barras apiladas al 100% mostrando la proporción de contratos
    con personas naturales vs jurídicas en cada grupo.
    Un alto porcentaje de personas naturales puede indicar
    dependencia de servicios personales.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con los datos de contrato (debe incluir 'grupo_comparacion', 'tipo_proveedor').
    colores : dict
        Diccionario de colores para la visualización.
    template : str
        Plantilla de Plotly para el estilo del gráfico.

    Retorna
    -------
    pd.DataFrame
        DataFrame agregado con la distribución porcentual por tipo de proveedor y grupo.
    """
    agg = (df.groupby(["grupo_comparacion", "tipo_proveedor"])
             .size().reset_index(name="contratos"))
    totales = df.groupby("grupo_comparacion").size().rename("total")
    agg = agg.merge(totales, on="grupo_comparacion")
    agg["pct"] = (agg["contratos"] / agg["total"] * 100).round(2)

    # Barras apiladas 100%
    fig = px.bar(
        agg,
        x="grupo_comparacion",
        y="pct",
        color="tipo_proveedor",
        barmode="relative",
        title="Distribución por tipo de proveedor (% del total de contratos)",
        labels={
            "pct": "% de contratos",
            "grupo_comparacion": "Grupo",
            "tipo_proveedor": "Tipo de proveedor"
        },
        text=agg["pct"].apply(lambda x: f"{x:.1f}%"),
        template=template
    )
    fig.update_traces(textposition="inside")
    fig.show()

    # Tabla de participación porcentual
    pivot = agg.pivot_table(index="tipo_proveedor", columns="grupo_comparacion",
                             values="pct", aggfunc="sum").fillna(0).round(2)
    print("\n Tabla de participación porcentual por tipo de proveedor:")
    display(pivot)
    return agg

agg_proveedor = graficar_tipo_proveedor(df, COLORES, TEMPLATE_PLOTLY)


 Tabla de participación porcentual por tipo de proveedor:


grupo_comparacion,Entidad X,Sector sin Entidad X
tipo_proveedor,,
CÉDULA DE CIUDADANÍA,93.38,87.50
CÉDULA DE EXTRANJERÍA,0.01,0.06
NIT,5.83,11.05
NO DEFINIDO,0.63,1.17
OTRO,0.14,0.19
PERMISO POR PROTECCIÓN TEMPORAL,0.00,0.01
TARJETA DE IDENTIDAD,0.00,0.01


### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'La Entidad X contrata un Y% de personas naturales, superando al sector en Z%, lo que podría sugerir una estrategia de contratación de servicios personales más intensiva o una dependencia de este tipo de fuerza de trabajo.']

In [60]:
def graficar_top_proveedores(df_entidad, colores, template, top_n=10):
    """
    Muestra los N proveedores más frecuentes de la Entidad X,
    tanto por número de contratos como por monto total.
    """
    # 1. Validación de datos
    if df_entidad.empty or "proveedor" not in df_entidad.columns:
        print(" Error: No hay datos disponibles para la Entidad X.")
        return

    df_plot = df_entidad.dropna(subset=["proveedor"]).copy()

    if df_plot.empty:
        print(" Advertencia: Todos los registros de proveedores son nulos (NaN).")
        print("Verifica la homologación de columnas en el Capítulo 4.")
        return

    # Por número de contratos
    top_contratos = (df_plot.groupby("proveedor")
                               .size().reset_index(name="contratos")
                               .nlargest(top_n, "contratos"))

    if not top_contratos.empty:
        fig1 = px.bar(top_contratos.sort_values("contratos"),
                      y="proveedor", x="contratos", orientation="h",
                      color_discrete_sequence=[colores["entidad"]],
                      title=f"Top {top_n} proveedores — Entidad X (por número de contratos)",
                      labels={"contratos": "Número de contratos", "proveedor": "Proveedor"},
                      text_auto=True, template=template)
        fig1.update_layout(yaxis=dict(autorange="reversed"))
        fig1.show()

    # Por monto total
    top_monto = (df_plot.groupby("proveedor")["valor_contrato_limpio"]
                           .sum().reset_index()
                           .nlargest(top_n, "valor_contrato_limpio"))
    top_monto["monto_mm"] = top_monto["valor_contrato_limpio"] / 1e9

    if not top_monto.empty:
        fig2 = px.bar(top_monto.sort_values("monto_mm"),
                      y="proveedor", x="monto_mm", orientation="h",
                      color_discrete_sequence=[colores["entidad"]],
                      title=f"Top {top_n} proveedores — Entidad X (por monto en miles de millones COP)",
                      labels={"monto_mm": "Monto (miles de millones COP)", "proveedor": "Proveedor"},
                      text_auto=".1f", template=template)
        fig2.update_layout(yaxis=dict(autorange="reversed"))
        fig2.show()

graficar_top_proveedores(df_entidad, COLORES, TEMPLATE_PLOTLY, top_n=10)

 Advertencia: Todos los registros de proveedores son nulos (NaN).
Verifica la homologación de columnas en el Capítulo 4.


###  Interpretación Pregunta 3
[Celda de interpretación: ¿Qué modalidad predomina en la Entidad X? ¿Hay exceso de contratación directa? ¿Hay dependencia de personas naturales? ¿Se concentra en pocos proveedores?]

---

## CAPÍTULO 9 — PREGUNTA 4: Patrones temporales y estadística descriptiva
> *¿Cuándo y cuánto? Analizando el ritmo y la magnitud de la contratación.*

---

In [61]:
def graficar_contratos_por_mes(df_entidad, colores, template):
    """
    Barras por mes mostrando el número de contratos y el monto contratado.
    Permite identificar meses con mayor concentración de actividad contractual.

    Parámetros
    ----------
    df_entidad : pd.DataFrame
        DataFrame con los contratos de la Entidad X.
    colores : dict
        Diccionario de colores para la visualización.
    template : str
        Plantilla de Plotly para el estilo del gráfico.

    Retorna
    -------
    pd.DataFrame
        DataFrame agregado con el número y monto de contratos por mes.
    """
    meses_nombres = {1:"Ene",2:"Feb",3:"Mar",4:"Abr",5:"May",6:"Jun",
                     7:"Jul",8:"Ago",9:"Sep",10:"Oct",11:"Nov",12:"Dic"}

    # Asegurar que 'mes' es numérico y manejar nulos
    df_plot = df_entidad.dropna(subset=['mes', 'valor_contrato_limpio']).copy()
    df_plot['mes'] = df_plot['mes'].astype(int)

    agg_mes = (df_plot.groupby("mes")
                         .agg(contratos=("valor_contrato_limpio", "count"),
                              monto=("valor_contrato_limpio", "sum"))
                         .reset_index())
    agg_mes["mes_nombre"] = agg_mes["mes"].map(meses_nombres)
    agg_mes["monto_mm"] = agg_mes["monto"] / 1e9

    fig = make_subplots(rows=1, cols=2,
                        subplot_titles=["Número de contratos por mes",
                                        "Monto contratado por mes (miles de millones COP)"])

    fig.add_bar(x=agg_mes["mes_nombre"], y=agg_mes["contratos"],
                marker_color=colores["entidad"],
                text=agg_mes["contratos"], textposition="outside",
                name="Contratos", row=1, col=1)

    fig.add_bar(x=agg_mes["mes_nombre"], y=agg_mes["monto_mm"],
                marker_color=colores["sector"], # Usando 'sector' para el monto, según la paleta
                text=agg_mes["monto_mm"].round(1), textposition="outside",
                name="Monto (MM COP)", row=1, col=2)

    fig.update_layout(title="Distribución mensual de contratación \u2014 Entidad X",
                      template=template, showlegend=False, height=450)
    fig.show()

    if not agg_mes.empty:
        mes_critico = agg_mes.loc[agg_mes["contratos"].idxmax(), "mes_nombre"]
        print(f"📅 Mes con mayor número de contratos: {mes_critico}")
    else:
        print("⚠️ No hay datos suficientes para determinar el mes con mayor número de contratos.")
    return agg_mes

agg_mes = graficar_contratos_por_mes(df_entidad, COLORES, TEMPLATE_PLOTLY)

📅 Mes con mayor número de contratos: Ene


### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'La contratación de la Entidad X muestra una clara estacionalidad, concentrándose en los meses de X e Y, tanto en número de contratos como en monto, lo que podría generar cuellos de botella administrativos.']

In [62]:
def calcular_estadisticas_descriptivas(df_entidad, df_sector):
    """
    Calcula y muestra la tabla completa de estadísticas descriptivas
    para el valor de los contratos, comparando Entidad X vs Sector.

    Incluye: n, media, mediana, mínimo, máximo, Q1, Q3, IQR,
    desviación estándar y coeficiente de variación.

    Parámetros
    ----------
    df_entidad : pd.DataFrame
        DataFrame con los contratos de la Entidad X.
    df_sector : pd.DataFrame
        DataFrame con los contratos del sector sin la Entidad X.

    Retorna
    -------
    pd.DataFrame
        Tabla con las estadísticas descriptivas calculadas.
    """
    def estadisticas(serie, nombre):
        if serie.dropna().empty:
            return {
                "Grupo": nombre,
                "n (contratos)": "0",
                "Media (COP)": "$0",
                "Mediana (COP)": "$0",
                "Mínimo (COP)": "$0",
                "Máximo (COP)": "$0",
                "Q1 (COP)": "$0",
                "Q3 (COP)": "$0",
                "IQR (COP)": "$0",
                "Desv. Estándar": "$0",
                "Coef. Variación (%)": "N/A"
            }
        else:
            q1  = serie.quantile(0.25)
            q3  = serie.quantile(0.75)
            iqr = q3 - q1
            cv  = (serie.std() / serie.mean() * 100) if serie.mean() != 0 else np.nan
            return {
                "Grupo": nombre,
                "n (contratos)": f"{serie.count():,}",
                "Media (COP)": f"${serie.mean():,.0f}",
                "Mediana (COP)": f"${serie.median():,.0f}",
                "Mínimo (COP)": f"${serie.min():,.0f}",
                "Máximo (COP)": f"${serie.max():,.0f}",
                "Q1 (COP)": f"${q1:,.0f}",
                "Q3 (COP)": f"${q3:,.0f}",
                "IQR (COP)": f"${iqr:,.0f}",
                "Desv. Estándar": f"${serie.std():,.0f}",
                "Coef. Variación (%)": f"{cv:.1f}%" if not pd.isna(cv) else "N/A"
            }

    tabla = pd.DataFrame([
        estadisticas(df_entidad["valor_contrato_limpio"].dropna(), "Entidad X"),
        estadisticas(df_sector["valor_contrato_limpio"].dropna(),  "Sector sin Entidad X")
    ])

    fig = go.Figure(data=[go.Table(
        header=dict(values=list(tabla.columns),
                    fill_color=COLORES["entidad"], font=dict(color="white", size=11),
                    align="center"),
        cells=dict(values=[tabla[c].tolist() for c in tabla.columns],
                   fill_color=[["#EBF5FB", "#D6EAF8"]],
                   align="center", font=dict(size=11))
    )])
    fig.update_layout(title="Estadísticas descriptivas del valor de contratos", height=220)
    fig.show()
    return tabla

tabla_stats = calcular_estadisticas_descriptivas(df_entidad, df_sector)

### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'La media del valor de los contratos en la Entidad X es significativamente superior a la mediana, lo que indica la presencia de contratos de alto valor que sesgan el promedio. El IQR muestra una gran dispersión en los valores de los contratos, sugiriendo una amplia gama de necesidades.']

In [63]:
def graficar_boxplot_comparativo(df, colores, template):
    """
    Boxplot del valor de los contratos por grupo de comparación.

    Cómo leer un boxplot:
    - La línea dentro de la caja: mediana (valor típico).
    - Los extremos de la caja: Q1 y Q3.
    - Los bigotes: 1.5 \u00D7 IQR desde los cuartiles.
    - Los puntos fuera de los bigotes: posibles valores atípicos.

    Se aplica escala logarítmica para manejar la alta dispersión
    típica en contratos públicos.

    Parámetros
    ----------
    df : pd.DataFrame
        DataFrame con los datos de contrato (debe incluir 'valor_contrato_limpio', 'grupo_comparacion').
    colores : dict
        Diccionario de colores para la visualización.
    template : str
        Plantilla de Plotly para el estilo del gráfico.
    """
    fig = px.box(
        df.dropna(subset=["valor_contrato_limpio"]), # Eliminar nulos para el boxplot
        x="grupo_comparacion",
        y="valor_contrato_limpio",
        color="grupo_comparacion",
        color_discrete_map={"Entidad X": colores["entidad"],
                             "Sector sin Entidad X": colores["sector"]},
        points="outliers", # Mostrar los puntos atípicos
        title="Distribución del valor de contratos \u2014 Entidad X vs Sector (escala log)",
        labels={
            "valor_contrato_limpio": "Valor del contrato (COP)",
            "grupo_comparacion": "Grupo"
        },
        log_y=True, # Aplicar escala logarítmica al eje Y
        template=template
    )
    fig.update_traces(boxmean=True)  # Agrega punto de media
    fig.show()

graficar_boxplot_comparativo(df, COLORES, TEMPLATE_PLOTLY)

### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'El boxplot confirma la alta dispersión de los valores de los contratos, con numerosos atípicos, especialmente en la Entidad X. La mediana, al estar más cerca del cuartil inferior, sugiere que la mayoría de los contratos tienen un valor relativamente bajo, mientras que unos pocos contratos de gran envergadura elevan la media.']

In [64]:
def identificar_atipicos(df_entidad, colores, template, top_n=10):
    """
    Identifica contratos que superan el limite superior del IQR.
    """
    serie = df_entidad["valor_contrato_limpio"].dropna()

    if serie.empty:
        print("Advertencia: No hay valores de contrato validos para identificar atipicos.")
        return pd.DataFrame()

    q1  = serie.quantile(0.25)
    q3  = serie.quantile(0.75)
    iqr = q3 - q1
    limite_superior = q3 + 1.5 * iqr

    atipicos = df_entidad[df_entidad["valor_contrato_limpio"] > limite_superior].copy()
    atipicos["monto_mm"] = atipicos["valor_contrato_limpio"] / 1e9

    print(f"Contratos atipicos (valor > ${limite_superior:,.0f}):")
    print(f"   Cantidad: {len(atipicos):,} contratos")

    if len(df_entidad) > 0:
        print(f"   Representan el {len(atipicos)/len(df_entidad)*100:.1f}% del total de contratos")

    if df_entidad['valor_contrato_limpio'].sum() > 0:
        print(f"   Concentran el {atipicos['valor_contrato_limpio'].sum()/df_entidad['valor_contrato_limpio'].sum()*100:.1f}% del monto total")

    if not atipicos.empty:
        fig = px.scatter(
            atipicos, x="fecha_contrato", y="monto_mm",
            color_discrete_sequence=[colores["alerta"]],
            hover_data={"tipo_contrato": True, "modalidad_contratacion": True, "proveedor": True, "valor_contrato_limpio": ':.0f'},
            title=f"Contratos atipicos - Entidad X (valor > ${limite_superior/1e9:.1f} mil millones COP)",
            labels={"monto_mm": "Monto (miles de millones COP)", "fecha_contrato": "Fecha"},
            template=template
        )
        fig.show()

        print(f"\n Top {top_n} contratos atipicos:")
        display(atipicos.nlargest(top_n, "valor_contrato_limpio")[
            ["entidad", "tipo_contrato", "modalidad_contratacion", "proveedor",
             "fecha_contrato", "monto_mm"]
        ])
    else:
        print("No se encontraron contratos atipicos segun el criterio definido.")

    return atipicos

### 🔍 Interpretación
[El equipo debe completar esta celda con los hallazgos específicos encontrados en los datos. Ejemplo: 'X contratos, que representan el Y% del total, son considerados atípicos por su alto valor. Estos contratos, a pesar de ser pocos, concentran Z% del monto total de contratación, lo que subraya la importancia de su gestión individualizada.']

In [65]:
def graficar_heatmap_mensual(df_entidad, template):
    """
    Heatmap de contratos: eje X = mes, eje Y = año.
    Muestra en qué combinaciones año-mes se concentra la actividad contractual.
    Útil para identificar patrones estacionales recurrentes.

    Parámetros
    ----------
    df_entidad : pd.DataFrame
        DataFrame con los contratos de la Entidad X.
    template : str
        Plantilla de Plotly para el estilo del gráfico.
    """
    # Asegurar que 'anio' y 'mes' son numéricos y manejar nulos
    df_plot = df_entidad.dropna(subset=['anio', 'mes']).copy()
    df_plot['anio'] = df_plot['anio'].astype(int)
    df_plot['mes'] = df_plot['mes'].astype(int)

    pivot = (df_plot.groupby([df_plot.anio, df_plot.mes])
                       .size().reset_index(name="contratos")
                       .pivot(index="anio", columns="mes", values="contratos")
                       .fillna(0))

    if pivot.empty:
        print("⚠️ No hay datos suficientes para generar el heatmap mensual.")
        return

    meses = ["Ene","Feb","Mar","Abr","May","Jun","Jul","Ago","Sep","Oct","Nov","Dic"]
    # Asegurarse de que las columnas de pivot coincidan con los meses antes de reasignar
    column_map = {m: meses[m-1] for m in pivot.columns if m in range(1,13)}
    pivot = pivot.rename(columns=column_map)

    # Asegurar el orden correcto de las columnas de meses
    ordered_columns = [m for m in meses if m in pivot.columns]
    pivot = pivot[ordered_columns]

    fig = px.imshow(
        pivot, text_auto=True,
        color_continuous_scale="Blues",
        title="Heatmap de actividad contractual \u2014 Entidad X (número de contratos por mes y año)",
        labels={
            "x": "Mes",
            "y": "Año",
            "color": "Contratos"
        }
    )
    fig.update_layout(template=template, xaxis=dict(side="top")) # Meses arriba
    fig.show()

graficar_heatmap_mensual(df_entidad, TEMPLATE_PLOTLY)

### 🔍 Interpretación Pregunta 4:
[Celda de interpretación. Incluir: ¿En qué meses se concentra la contratación? ¿La media supera significativamente a la mediana (sesgo por atípicos)? ¿Qué dice el IQR sobre la dispersión? ¿Qué proporción de contratos son atípicos y cuánto concentran del monto?]

---

## CAPÍTULO 10 — Conclusiones y acciones sugeridas
> *De la evidencia a las decisiones: acciones para la planeación contractual.*

---

Este capítulo final cierra nuestra historia de análisis de datos de contratación pública. Aquí, consolidaremos los hallazgos más relevantes de los capítulos anteriores y, lo más importante, los traduciremos en acciones concretas y sugerencias para mejorar la planeación contractual de la Entidad X. Es el puente entre el análisis de datos y la toma de decisiones estratégicas. El objetivo es que la Entidad X pueda usar esta evidencia para optimizar sus procesos, fomentar la transparencia y asegurar una gestión más eficiente de sus recursos.

### Hallazgo 1 — Comportamiento económico:
[Escribir con base en los resultados de la Pregunta 2]

### Hallazgo 2 — Estilo de contratación:
[Escribir con base en la Pregunta 3]

### Hallazgo 3 — Temporalidad y distribución:
[Escribir con base en la Pregunta 4]

### Acción sugerida 1:
[Propuesta concreta de mejora en planeación]

### Acción sugerida 2:
[Propuesta concreta de mejora en planeación]

### Acción sugerida 3:
[Propuesta concreta de mejora en planeación]

In [66]:
def dashboard_ejecutivo(df_entidad, df_sector, colores, template):
    """
    Panel ejecutivo de 4 indicadores clave en una sola visualización:
    - Total contratos Entidad X
    - Monto total Entidad X
    - Modalidad más usada
    - Mes con mayor actividad
    Ideal para la primera diapositiva de la presentación ejecutiva.

    Parámetros
    ----------
    df_entidad : pd.DataFrame
        DataFrame con los contratos de la Entidad X.
    df_sector : pd.DataFrame
        DataFrame con los contratos del sector sin la Entidad X (no utilizado directamente en los indicadores, pero mantiene la firma).
    colores : dict
        Diccionario de colores para la visualización.
    template : str
        Plantilla de Plotly para el estilo del gráfico.
    """
    meses_nombres = {1:"Ene",2:"Feb",3:"Mar",4:"Abr",5:"May",6:"Jun",
                     7:"Jul",8:"Ago",9:"Sep",10:"Oct",11:"Nov",12:"Dic"}

    total_contratos = len(df_entidad)
    monto_total     = df_entidad["valor_contrato_limpio"].sum() / 1e9 if not df_entidad["valor_contrato_limpio"].empty else 0
    modalidad_top   = df_entidad["modalidad_contratacion"].value_counts().idxmax() if not df_entidad["modalidad_contratacion"].empty else "N/A"
    mes_top_val = df_entidad["mes"].value_counts().idxmax() if not df_entidad["mes"].empty else None
    mes_top         = meses_nombres.get(int(mes_top_val), "N/A") if mes_top_val is not None else "N/A"

    fig = go.Figure()
    indicadores = [
        {"valor": f"{total_contratos:,}", "titulo": "Total contratos<br>Entidad X", "color": colores["entidad"]},
        {"valor": f"${monto_total:.1f} MM", "titulo": "Monto total<br>(miles de millones COP)", "color": colores["entidad"]},
        {"valor": modalidad_top[:25], "titulo": "Modalidad<br>predominante", "color": colores["sector"]},
        {"valor": mes_top, "titulo": "Mes de mayor<br>actividad", "color": colores["acento"]}
    ]

    for i, ind in enumerate(indicadores):
        fig.add_annotation(
            x=i*0.33, y=0.5, xref="paper", yref="paper",
            text=f"<b style='font-size:28px;color:{ind['color']}'>{ind['valor']}</b><br>"+
                 f"<span style='font-size:13px;color:#555'>{ind['titulo']}</span>",
            showarrow=False, align="center"
        )

    fig.update_layout(
        title="Dashboard ejecutivo \u2014 Entidad X (Alcaldía, Santander)",
        height=250, template=template,
        xaxis=dict(visible=False), yaxis=dict(visible=False)
    )
    fig.show()

dashboard_ejecutivo(df_entidad, df_sector, COLORES, TEMPLATE_PLOTLY)

## Estructura Sugerida para la Presentación Ejecutiva (5 diapositivas)

### Diapositiva 1: Resumen Ejecutivo
*   **Título:** 'Visión General de la Contratación Pública en [Nombre de la Entidad X]'
*   **Contenido:** Dashboard ejecutivo final (la última gráfica generada) con los 4 KPIs clave.
*   **Responsables:** [Nombres de los integrantes del equipo]

### Diapositiva 2: Comportamiento en Monto y Volumen (Pregunta 2)
*   **Título:** 'Análisis de la Evolución Anual de la Contratación'
*   **Contenido:** Las dos gráficas de evolución anual (monto y número de contratos) con su respectiva interpretación.
*   **Responsables:** [Nombres de los integrantes del equipo]

### Diapositiva 3: Estilo de Contratación (Pregunta 3)
*   **Título:** 'Modalidades y Proveedores: El 'Cómo' de la Contratación'
*   **Contenido:** Gráficas de distribución por modalidad de contratación y tipo de proveedor, con su interpretación.
*   **Responsables:** [Nombres de los integrantes del equipo]

### Diapositiva 4: Patrones Temporales y Atípicos (Pregunta 4)
*   **Título:** 'Ritmo de Contratación y Detección de Valores Atípicos'
*   **Contenido:** Heatmap de actividad mensual, tabla de estadísticas descriptivas o boxplot comparativo, e identificación de contratos atípicos. Incluir su interpretación.
*   **Responsables:** [Nombres de los integrantes del equipo]

### Diapositiva 5: Conclusiones y Acciones Estratégicas
*   **Título:** 'Hallazgos Clave y Recomendaciones para la Planeación'
*   **Contenido:** Presentación de los tres hallazgos clave y las tres acciones sugeridas del Capítulo 10.
*   **Responsables:** [Nombres de los integrantes del equipo]

In [67]:
# Load data from the secop_contratos_santander table
query = "SELECT * FROM secop_contratos_santander"
df_contratos = pd.read_sql(query, engine)

# Display the first 5 rows of the DataFrame
display(df_contratos.head())

,nombre_entidad,nit_entidad,departamento,ciudad,localizaci_n,orden,sector,rama,entidad_centralizada,proceso_de_compra,...,nombre_ordenador_de_pago,tipo_de_documento_ordenador_de_pago,n_mero_de_documento_ordenador_de_pago,documentos_tipo,descripcion_documentos_tipo,fecha_de_firma,ultima_actualizacion,fecha_inicio_liquidacion,fecha_fin_liquidacion,fecha_de_notificaci_n_de_prorrogaci_n
0,MUNICIPIO DE PIEDECUESTA,890205383,Santander,Piedecuesta,"Colombia, Santander , Piedecuesta",Territorial,No aplica/No pertenece,Ejecutivo,Centralizada,CO1.BDOS.4644540,...,WILLIAM JAVIER NIÑO ACEVEDO,Cédula de Ciudadanía,91347842,No,No definido,2023-06-27T00:00:00.000,None,None,None,None
1,ALCALDIA DISTRITAL BARRANCABERMEJA,890201900,Santander,No Definido,"Colombia, Santander , No Definido",Territorial,Servicio Público,Ejecutivo,Centralizada,CO1.BDOS.4644739,...,ISOLINA TIRADO SILVA,Cédula de Ciudadanía,37933219,No,No definido,2023-06-28T00:00:00.000,2024-03-14T00:00:00.000,None,None,None
2,MUNICIPIO DE BUCARAMANGA,890201222,Santander,Bucaramanga,"Colombia, Santander , Bucaramanga",Territorial,Servicio Público,Ejecutivo,Descentralizada,CO1.BDOS.4644827,...,No definido,No definido,No definido,No,No definido,2023-06-27T00:00:00.000,2023-11-27T00:00:00.000,None,None,2023-11-20T00:00:00.000
3,ALCALDIA DISTRITAL BARRANCABERMEJA,890201900,Santander,No Definido,"Colombia, Santander , No Definido",Territorial,Servicio Público,Ejecutivo,Centralizada,CO1.BDOS.4644579,...,No definido,No definido,No definido,No,No definido,2023-06-28T00:00:00.000,None,None,None,None
4,ALCALDÍA MUNICIPAL DE EL SOCORRO,890203688,Santander,Socorro,"Colombia, Santander , Socorro",Territorial,Servicio Público,Ejecutivo,Centralizada,CO1.BDOS.4644776,...,No definido,No definido,No definido,No,No definido,2023-06-28T00:00:00.000,2023-12-20T00:00:00.000,None,None,None
